# 🚀 Multimodal Agents Lab (Enhanced Edition)

## Build Multimodal AI Agents with MongoDB and Google Gemini

This enhanced version includes:
- ✅ Real-time validation for each CODE_BLOCK
- 📊 Visual progress tracking
- 💡 Helpful hints when you're stuck
- 🎯 Clear success indicators
- 🏆 Achievement system

### What You'll Build
A multimodal AI agent that can:
- Process PDF documents and extract images
- Create vector embeddings for semantic search
- Answer questions using AI tools
- Remember conversation history
- Reason and act using the ReAct pattern

## 🎮 Load the Validation System

In [2]:
# Load validation system
import inspect
import os
import dotenv
import time
from typing import Any, Tuple, List, Dict, Callable
from datetime import datetime
from IPython.display import display, HTML, Image as IPImage
import ipywidgets as widgets

class ValidationResult:
    """Represents the result of a validation check"""
    def __init__(self, passed: bool, message: str, hints: List[str] = None, details: Dict = None):
        self.passed = passed
        self.message = message
        self.hints = hints or []
        self.details = details or {}
        self.timestamp = datetime.now()
    
    def display(self):
        """Display validation result with formatting"""
        if self.passed:
            status_html = '<div style="padding: 10px; background-color: #d4edda; border: 1px solid #c3e6cb; border-radius: 5px; margin: 10px 0;">'
            status_html += '<h3 style="color: #155724;">✅ Validation Passed!</h3>'
            status_html += f'<p style="color: #155724;">{self.message}</p>'
        else:
            status_html = '<div style="padding: 10px; background-color: #f8d7da; border: 1px solid #f5c6cb; border-radius: 5px; margin: 10px 0;">'
            status_html += '<h3 style="color: #721c24;">❌ Validation Failed</h3>'
            status_html += f'<p style="color: #721c24;">{self.message}</p>'
            
            if self.hints:
                status_html += '<h4 style="color: #856404;">💡 Hints:</h4>'
                status_html += '<ul style="color: #856404;">'
                for hint in self.hints:
                    status_html += f'<li>{hint}</li>'
                status_html += '</ul>'
        
        status_html += '</div>'
        display(HTML(status_html))

class LabProgress:
    """Tracks student progress through the lab"""
    def __init__(self):
        self.completed_blocks = set()
        self.attempts = {}
        self.start_time = datetime.now()
    
    def record_attempt(self, block_num: int, passed: bool):
        if block_num not in self.attempts:
            self.attempts[block_num] = []
        self.attempts[block_num].append({
            'passed': passed,
            'timestamp': datetime.now()
        })
        if passed:
            self.completed_blocks.add(block_num)
            # Check achievements
            achievements.check_achievements(self.completed_blocks)
    
    def display_progress(self, total_blocks: int = 19):
        percentage = (len(self.completed_blocks) / total_blocks) * 100
        completed = len(self.completed_blocks)
        
        progress_html = f'''
        <div style="margin: 20px 0;">
            <h3>📊 Your Progress: {completed}/{total_blocks} CODE_BLOCKs completed ({percentage:.1f}%)</h3>
            <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; overflow: hidden;">
                <div style="width: {percentage}%; background-color: #4CAF50; height: 30px; text-align: center; line-height: 30px; color: white;">
                    {percentage:.1f}%
                </div>
            </div>
        </div>
        '''
        display(HTML(progress_html))

class AchievementSystem:
    """Gamification through achievements"""
    
    def __init__(self):
        self.achievements = {
            'first_step': {'name': '🎉 First Steps', 'desc': 'Complete your first CODE_BLOCK', 'blocks_needed': 1},
            'pdf_master': {'name': '📄 PDF Master', 'desc': 'Complete all PDF processing blocks', 'blocks_needed': [1, 2]},
            'data_engineer': {'name': '💾 Data Engineer', 'desc': 'Successfully store data in MongoDB', 'blocks_needed': [3]},
            'search_wizard': {'name': '🔍 Search Wizard', 'desc': 'Implement vector search', 'blocks_needed': [4, 5, 6]},
            'tool_builder': {'name': '🛠️ Tool Builder', 'desc': 'Create all agent tools', 'blocks_needed': [7, 8, 9]},
            'ai_engineer': {'name': '🤖 AI Engineer', 'desc': 'Connect LLM with tools', 'blocks_needed': [10, 11, 12]},
            'memory_keeper': {'name': '🧠 Memory Keeper', 'desc': 'Implement conversation memory', 'blocks_needed': [13, 14, 15, 16, 17, 18, 19]},
            'lab_champion': {'name': '🏆 Lab Champion', 'desc': 'Complete all CODE_BLOCKs!', 'blocks_needed': list(range(1, 20))}
        }
        self.earned = set()
    
    def check_achievements(self, completed_blocks: set):
        """Check for newly earned achievements"""
        newly_earned = []
        
        for ach_id, ach_data in self.achievements.items():
            if ach_id not in self.earned:
                blocks_needed = ach_data['blocks_needed']
                
                if isinstance(blocks_needed, int):
                    if len(completed_blocks) >= blocks_needed:
                        self.earned.add(ach_id)
                        newly_earned.append(ach_data)
                else:
                    if all(b in completed_blocks for b in blocks_needed):
                        self.earned.add(ach_id)
                        newly_earned.append(ach_data)
        
        for achievement in newly_earned:
            self._display_achievement(achievement)
    
    def _display_achievement(self, achievement: dict):
        """Display achievement notification"""
        html = f"""
        <div style="
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        ">
            <h2 style="margin: 0; font-size: 48px;">🎊</h2>
            <h3 style="margin: 10px 0;">Achievement Unlocked!</h3>
            <h2 style="margin: 10px 0;">{achievement['name']}</h2>
            <p style="margin: 5px 0; opacity: 0.9;">{achievement['desc']}</p>
        </div>
        """
        display(HTML(html))

# Global instances
lab_progress = LabProgress()
achievements = AchievementSystem()

def create_validation_widget(block_num: int, validator_func: Callable, variable_name: str):
    """Creates an interactive validation widget for a CODE_BLOCK"""
    
    button = widgets.Button(
        description=f'Validate CODE_BLOCK_{block_num}',
        button_style='primary',
        icon='check'
    )
    
    output = widgets.Output()
    
    def on_click(b):
        with output:
            output.clear_output()
            
            try:
                var = globals().get(variable_name)
                if var is None:
                    result = ValidationResult(
                        False,
                        f"Variable '{variable_name}' not found.",
                        hints=[f"Make sure you've run the CODE_BLOCK_{block_num} cell and created '{variable_name}'"]
                    )
                else:
                    result = validator_func(var)
                    lab_progress.record_attempt(block_num, result.passed)
                
                result.display()
                
                if result.passed:
                    lab_progress.display_progress()
                    
            except Exception as e:
                result = ValidationResult(
                    False,
                    f"Error during validation: {str(e)}",
                    hints=["Check your code for syntax errors"]
                )
                result.display()
    
    button.on_click(on_click)
    
    return widgets.VBox([button, output])

print("✅ Validation system loaded successfully!")
lab_progress.display_progress()

✅ Validation system loaded successfully!


## 📋 Prerequisites and Setup

In [3]:
# TODO: Set your environment variables here
# You can get these from:
# - Google AI Studio: https://makersuite.google.com/app/apikey
# - VoyageAI: https://www.voyageai.com/
# - MongoDB Atlas: https://www.mongodb.com/cloud/atlas

from dotenv import load_dotenv

import os
load_dotenv()

# os.environ["GOOGLE_API_KEY"] = "your-google-api-key"
# os.environ["VOYAGE_API_KEY"] = "your-voyage-api-key"
# os.environ["MONGO_URI"] = "your-mongodb-connection-string"

True

## Step 1: Connect to MongoDB

In [4]:
# Import required libraries
import os
from pymongo import MongoClient

# Connect to MongoDB
MONGO_URI = os.environ.get("MONGO_URI")
if not MONGO_URI:
    raise ValueError("Please set the MONGO_URI environment variable")

mongodb_client = MongoClient(MONGO_URI)
db = mongodb_client["multimodal_agents_lab"]
collection = db["embeddings"]

# Test the connection
print(f"Connected to MongoDB! Database: {db.name}")
print(f"Collection document count: {collection.count_documents({})}")

Connected to MongoDB! Database: multimodal_agents_lab
Collection document count: 0


## Step 2: Download and Process a PDF

We'll download a research paper and extract its pages as images.

In [5]:
# Download the DeepSeek paper from arxiv
import requests

pdf_url = "https://arxiv.org/pdf/2411.04760"
response = requests.get(pdf_url)
pdf_stream = response.content

print(f"Downloaded PDF: {len(pdf_stream):,} bytes")

Downloaded PDF: 2,795,836 bytes


In [7]:
# Now let's open the PDF using PyMuPDF
import pymupdf

# CODE_BLOCK_1: Open the PDF from the byte stream
# Hint: Use pymupdf.Document() with stream and filetype parameters
# <CODE_BLOCK_1>
pdf = pymupdf.Document(stream=pdf_stream, filetype="pdf")

In [8]:
# Validation for CODE_BLOCK_1
def validate_code_block_1(pdf_doc):
    try:
        if not hasattr(pdf_doc, 'page_count'):
            return ValidationResult(
                False,
                "The pdf doesn't appear to be a valid PyMuPDF Document object.",
                hints=[
                    "Use pymupdf.Document() to create the document",
                    "Pass stream=pdf_stream as the first parameter",
                    "Pass filetype='pdf' as the second parameter"
                ]
            )
        
        if pdf_doc.page_count == 0:
            return ValidationResult(
                False,
                "The PDF document has no pages.",
                hints=["Verify that pdf_stream contains the PDF data"]
            )
        
        return ValidationResult(
            True,
            f"Perfect! PDF opened successfully with {pdf_doc.page_count} pages.",
            details={'page_count': pdf_doc.page_count}
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error during validation: {str(e)}",
            hints=["Check if 'pdf' variable is defined", "Ensure pymupdf is imported"]
        )

display(create_validation_widget(1, validate_code_block_1, 'pdf'))

## Step 3: Extract Pages as Images

In [9]:
# Create directory for images
import os
os.makedirs("data/images", exist_ok=True)

# Process each page
docs = []
zoom = 0.5  # Reduced to make images smaller
mat = pymupdf.Matrix(zoom, zoom)

for n in range(len(pdf)):
    page = pdf[n]
    
    # CODE_BLOCK_2: Render the page as a pixmap with the zoom matrix
    # Hint: Use page.get_pixmap() with the matrix parameter
    # <CODE_BLOCK_2>
    pix = pdf[n].get_pixmap(matrix=mat)

    # Save the image
    img_path = f"data/images/{n+1}.png"
    pix.save(img_path)
    
    # Store metadata
    docs.append({
        "page_number": n + 1,
        "width": pix.width,
        "height": pix.height,
        "image_path": img_path
    })
    
    print(f"Processed page {n+1}/{len(pdf)}")

print(f"\nExtracted {len(docs)} pages as images")

Processed page 1/19
Processed page 2/19
Processed page 3/19
Processed page 4/19
Processed page 5/19
Processed page 6/19
Processed page 7/19
Processed page 8/19
Processed page 9/19
Processed page 10/19
Processed page 11/19
Processed page 12/19
Processed page 13/19
Processed page 14/19
Processed page 15/19
Processed page 16/19
Processed page 17/19
Processed page 18/19
Processed page 19/19

Extracted 19 pages as images


In [10]:
# Validation for CODE_BLOCK_2
def validate_code_block_2(pixmap):
    try:
        # Check if pix exists and has expected attributes
        if not hasattr(pixmap, 'width') or not hasattr(pixmap, 'height'):
            return ValidationResult(
                False,
                "The pix object doesn't have the expected attributes.",
                hints=[
                    "Use page.get_pixmap() to create the pixmap",
                    "Pass matrix=mat as the parameter",
                    "Make sure the variable is named 'pix'"
                ]
            )
        
        # Check if images were saved
        if not os.path.exists("data/images/1.png"):
            return ValidationResult(
                False,
                "Images were not saved to the expected location.",
                hints=["Make sure the loop completed successfully"]
            )
        
        return ValidationResult(
            True,
            f"Excellent! Pages rendered successfully. Image size: {pixmap.width}x{pixmap.height} pixels."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error during validation: {str(e)}",
            hints=["Check if 'pix' is defined in the loop", "Ensure the loop ran successfully"]
        )

# Note: We validate using the last pix from the loop
display(create_validation_widget(2, validate_code_block_2, 'pix'))

## Step 4: Generate Multimodal Embeddings

For this lab, we'll use pre-generated embeddings to save time and API costs.

In [11]:
# Load pre-generated embeddings
import json

with open("data/embeddings.json", "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} documents with embeddings")
print(f"Embedding dimension: {len(data[0]['embedding'])}")
print(f"Sample document keys: {list(data[0].keys())}")

Loaded 22 documents with embeddings
Embedding dimension: 1024
Sample document keys: ['key', 'width', 'height', 'embedding']


## Step 5: Store Data in MongoDB

In [13]:
# Clear existing data
collection.delete_many({})

# CODE_BLOCK_3: Insert all documents into MongoDB
# Hint: Use collection.insert_many() with the data list
# <CODE_BLOCK_3>
collection.insert_many(data)
print(f"Inserted {collection.count_documents({})} documents")

Inserted 22 documents


In [14]:
# Validation for CODE_BLOCK_3
def validate_code_block_3(insert_result):
    try:
        # Check document count in collection
        count = collection.count_documents({})
        
        if count == 0:
            return ValidationResult(
                False,
                "No documents were inserted into the collection.",
                hints=[
                    "Use collection.insert_many(data)",
                    "Make sure 'data' contains the documents to insert"
                ]
            )
        
        if count != len(data):
            return ValidationResult(
                False,
                f"Expected {len(data)} documents but found {count}.",
                hints=["Make sure you inserted all documents from the data list"]
            )
        
        # Check document structure
        sample = collection.find_one()
        required_fields = ['embedding', 'key', 'width', 'height']
        missing_fields = [f for f in required_fields if f not in sample]
        
        if missing_fields:
            return ValidationResult(
                False,
                f"Documents are missing required fields: {missing_fields}",
                hints=["Make sure you're inserting the correct data"]
            )
        
        return ValidationResult(
            True,
            f"Excellent! Successfully inserted {count} documents with embeddings into MongoDB."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error during validation: {str(e)}",
            hints=["Check MongoDB connection", "Ensure insert_many was called"]
        )

# Create a dummy insert_result for validation
insert_result = True  # The actual result would come from insert_many
display(create_validation_widget(3, validate_code_block_3, 'insert_result'))

## Step 6: Create Vector Search Index

In [ ]:
from pymongo.operations import SearchIndexModel

# Define the vector search index
VS_INDEX_NAME = "vector_index"

model = SearchIndexModel(
    definition={
        "mappings": {
            "dynamic": True,
            "fields": {
                "embedding": {
                    "type": "knnVector",
                    "dimensions": 1024,
                    "similarity": "cosine",
                }
            },
        }
    },
    name=VS_INDEX_NAME,
)

# Check if index already exists
existing_indexes = list(collection.list_search_indexes())
if any(idx['name'] == VS_INDEX_NAME for idx in existing_indexes):
    print(f"Index '{VS_INDEX_NAME}' already exists")
else:
    # CODE_BLOCK_4: Create the vector search index
    # Hint: Use collection.create_search_index() with the model
    <CODE_BLOCK_4>
    print(f"Creating index '{VS_INDEX_NAME}'...")

In [ ]:
# Validation for CODE_BLOCK_4
def validate_code_block_4(index_name):
    try:
        # Wait a moment for index to be created
        time.sleep(2)
        
        # Check if index exists
        indexes = list(collection.list_search_indexes())
        index_names = [idx['name'] for idx in indexes]
        
        if VS_INDEX_NAME not in index_names:
            return ValidationResult(
                False,
                f"Index '{VS_INDEX_NAME}' was not found.",
                hints=[
                    "Use collection.create_search_index(model)",
                    "Make sure 'model' is defined with the index configuration"
                ]
            )
        
        # Find our index
        our_index = next(idx for idx in indexes if idx['name'] == VS_INDEX_NAME)
        
        return ValidationResult(
            True,
            f"Great! Vector search index '{VS_INDEX_NAME}' created successfully. Status: {our_index.get('status', 'CREATING')}"
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error during validation: {str(e)}",
            hints=["Check if the index creation was successful"]
        )

# Use VS_INDEX_NAME for validation
display(create_validation_widget(4, validate_code_block_4, 'VS_INDEX_NAME'))

## Step 7: Implement Vector Search

In [ ]:
# Wait for index to be ready
import time

print("Waiting for index to be ready...")
while True:
    indexes = list(collection.list_search_indexes())
    our_index = next((idx for idx in indexes if idx['name'] == VS_INDEX_NAME), None)
    if our_index and our_index.get('status') == 'READY':
        print("Index is ready!")
        break
    time.sleep(5)

In [ ]:
# Test vector search
import voyageai

# Initialize Voyage client
voyage_client = voyageai.Client(api_key=os.environ.get("VOYAGE_API_KEY"))

# Create a test query
query = "scaling laws and compute efficiency"
query_embedding = voyage_client.embed(
    texts=[query], 
    model="voyage-3", 
    input_type="query"
).embeddings[0]

# CODE_BLOCK_5: Define the aggregation pipeline for vector search
# The pipeline should:
# 1. Use $vectorSearch stage with index name, path, queryVector, numCandidates=150, limit=2
# 2. Use $project stage to return: key, width, height, and score (using $meta: "vectorSearchScore")
pipeline = <CODE_BLOCK_5>

# CODE_BLOCK_6: Execute the aggregation pipeline
# Hint: Use collection.aggregate() with the pipeline
results = <CODE_BLOCK_6>

# Display results
result_list = list(results)
print(f"Found {len(result_list)} results:")
for i, result in enumerate(result_list):
    print(f"\nResult {i+1}:")
    print(f"  Image: {result['key']}")
    print(f"  Score: {result['score']:.4f}")

In [ ]:
# Validation for CODE_BLOCK_5
def validate_code_block_5(pipeline):
    try:
        if not isinstance(pipeline, list):
            return ValidationResult(
                False,
                "Pipeline should be a list of stages.",
                hints=["Create a list with dictionary stages"]
            )
        
        if len(pipeline) != 2:
            return ValidationResult(
                False,
                f"Pipeline has {len(pipeline)} stages, but should have 2.",
                hints=["Include both $vectorSearch and $project stages"]
            )
        
        # Check first stage
        if '$vectorSearch' not in pipeline[0]:
            return ValidationResult(
                False,
                "First stage should be $vectorSearch.",
                hints=["Start with {'$vectorSearch': {...}}"]
            )
        
        # Check second stage
        if '$project' not in pipeline[1]:
            return ValidationResult(
                False,
                "Second stage should be $project.",
                hints=["Add {'$project': {...}} as second stage"]
            )
        
        # Check vector search configuration
        vs_config = pipeline[0]['$vectorSearch']
        required_fields = ['index', 'path', 'queryVector', 'numCandidates', 'limit']
        missing = [f for f in required_fields if f not in vs_config]
        
        if missing:
            return ValidationResult(
                False,
                f"$vectorSearch is missing fields: {missing}",
                hints=[
                    "index: VS_INDEX_NAME",
                    "path: 'embedding'",
                    "queryVector: query_embedding",
                    "numCandidates: 150",
                    "limit: 2"
                ]
            )
        
        return ValidationResult(
            True,
            "Perfect! Your vector search pipeline is correctly configured."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error validating pipeline: {str(e)}",
            hints=["Check pipeline syntax"]
        )

display(create_validation_widget(5, validate_code_block_5, 'pipeline'))

In [ ]:
# Validation for CODE_BLOCK_6
def validate_code_block_6(results):
    try:
        # Convert cursor to list for validation
        if hasattr(results, '__iter__'):
            result_list = list(results)
        else:
            return ValidationResult(
                False,
                "Results should be an iterable (cursor) from aggregate().",
                hints=["Use collection.aggregate(pipeline)"]
            )
        
        if len(result_list) == 0:
            return ValidationResult(
                False,
                "No results returned from search.",
                hints=["Check if the pipeline is correct", "Ensure index is ready"]
            )
        
        # Check result structure
        required_fields = ['key', 'width', 'height', 'score']
        missing = [f for f in required_fields if f not in result_list[0]]
        
        if missing:
            return ValidationResult(
                False,
                f"Results are missing fields: {missing}",
                hints=["Check your $project stage"]
            )
        
        return ValidationResult(
            True,
            f"Excellent! Vector search returned {len(result_list)} results with scores."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error validating results: {str(e)}",
            hints=["Make sure results is defined"]
        )

# Note: results was consumed by list(), so we validate result_list
display(create_validation_widget(6, validate_code_block_6, 'result_list'))

## Step 8: Create AI Agent Tools

In [ ]:
# Define tool functions
def semantic_search(query: str, limit: int = 2) -> list:
    """Search for similar images using semantic search"""
    # Generate embedding for query
    query_embedding = voyage_client.embed(
        texts=[query], 
        model="voyage-3", 
        input_type="query"
    ).embeddings[0]
    
    # Search pipeline
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 150,
                "limit": limit,
            }
        },
        {
            "$project": {
                "_id": 0,
                "key": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]
    
    results = list(collection.aggregate(pipeline))
    return [result['key'] for result in results]

def query_mongodb(query: dict) -> list:
    """Query MongoDB with a filter"""
    results = list(collection.find(query, {"_id": 0, "key": 1}).limit(5))
    return [result['key'] for result in results]

# Combined tool for the agent
def get_information_for_question_answering(user_query: str) -> list:
    """Get relevant images to answer a user's question"""
    return semantic_search(user_query, limit=3)

# Test the tool
test_images = get_information_for_question_answering("What are the scaling laws?")
print(f"Found {len(test_images)} relevant images: {test_images}")

## Step 9: Configure Google Gemini

In [ ]:
# Initialize Gemini
import google.genai as genai

gemini_client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))
LLM = "gemini-2.0-flash-exp"

# Define function declarations for Gemini
get_information_declaration = genai.protos.FunctionDeclaration(
    name=<CODE_BLOCK_7>,  # The function name as a string
    description="Get relevant page images from a multimodal knowledge base to answer questions",
    parameters=genai.protos.Schema(
        type=genai.protos.Type.OBJECT,
        properties={
            "user_query": genai.protos.Schema(
                type=<CODE_BLOCK_8>,  # The parameter type
                description="The user's question or query"
            ),
        },
        required=<CODE_BLOCK_9>  # List of required parameters
    )
)

print("Function declaration created successfully!")

In [ ]:
# Validation for CODE_BLOCK_7, 8, 9
def validate_code_block_789(declaration):
    try:
        if declaration.name != "get_information_for_question_answering":
            return ValidationResult(
                False,
                f"Function name is '{declaration.name}' but should be 'get_information_for_question_answering'.",
                hints=["CODE_BLOCK_7 should be the exact function name as a string"]
            )
        
        # Check parameter type
        param_type = declaration.parameters.properties['user_query'].type
        if param_type != genai.protos.Type.STRING:
            return ValidationResult(
                False,
                "Parameter type is incorrect.",
                hints=["CODE_BLOCK_8 should be genai.protos.Type.STRING"]
            )
        
        # Check required parameters
        if declaration.parameters.required != ["user_query"]:
            return ValidationResult(
                False,
                f"Required parameters are {declaration.parameters.required} but should be ['user_query'].",
                hints=["CODE_BLOCK_9 should be ['user_query']"]
            )
        
        return ValidationResult(
            True,
            "Perfect! Function declaration is correctly configured for Gemini."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error validating declaration: {str(e)}",
            hints=["Check all three CODE_BLOCKs (7, 8, 9)"]
        )

# Combined validation for blocks 7-9
display(create_validation_widget(7, validate_code_block_789, 'get_information_declaration'))

## Step 10: Implement Tool Calling

In [ ]:
# Configure tools
tools_config = genai.protos.GenerateContentConfig(
    tools=[genai.protos.Tool(function_declarations=[get_information_declaration])],
    temperature=0.25,
)

# Function to select tools based on user query
def select_tool(messages):
    """Use Gemini to select the appropriate tool"""
    contents = messages
    
    # CODE_BLOCK_10: Generate response with tool configuration
    # Hint: Use gemini_client.models.generate_content() with model, contents, and config parameters
    response = <CODE_BLOCK_10>
    
    # Extract function call if present
    for part in response.candidates[0].content.parts:
        if hasattr(part, 'function_call'):
            return part.function_call
    
    return None

In [ ]:
# Validation for CODE_BLOCK_10
def validate_code_block_10(select_func):
    try:
        # Test the function
        test_query = "What are the main findings about scaling laws?"
        result = select_func([test_query])
        
        if result is None:
            return ValidationResult(
                False,
                "Function didn't return a tool call.",
                hints=[
                    "Use gemini_client.models.generate_content()",
                    "Pass model=LLM",
                    "Pass contents=contents",
                    "Pass config=tools_config"
                ]
            )
        
        if not hasattr(result, 'name') or not hasattr(result, 'args'):
            return ValidationResult(
                False,
                "Result doesn't have expected function call structure.",
                hints=["Check that the response contains function_call"]
            )
        
        return ValidationResult(
            True,
            f"Excellent! Gemini selected tool '{result.name}' with args: {dict(result.args)}"
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error testing function: {str(e)}",
            hints=["Check CODE_BLOCK_10 syntax"]
        )

display(create_validation_widget(10, validate_code_block_10, 'select_tool'))

## Step 11: Build the Basic Agent

In [ ]:
# Agent function
def agent(user_query: str):
    """Process user query and return answer with relevant images"""
    print(f"User: {user_query}")
    
    # CODE_BLOCK_11: Call select_tool with the user query
    # Hint: Pass [user_query] as a list
    tool_call = <CODE_BLOCK_11>
    
    if tool_call:
        print(f"\nSelected tool: {tool_call.name}")
        print(f"Arguments: {dict(tool_call.args)}")
        
        # CODE_BLOCK_12: Execute the tool function with the arguments
        # Hint: Use get_information_for_question_answering() with unpacked args
        tool_images = <CODE_BLOCK_12>
        
        print(f"\nRetrieved {len(tool_images)} relevant images")
        
        # Load images for context
        from PIL import Image
        images = [Image.open(img_path) for img_path in tool_images]
        
        # Generate final answer
        messages = [user_query] + images
        response = gemini_client.models.generate_content(
            model=LLM,
            contents=messages,
        )
        
        answer = response.text
        print(f"\nAgent: {answer}")
        
        # Display images
        for img_path in tool_images:
            display(IPImage(filename=img_path, width=400))
        
        return answer, tool_images
    else:
        print("\nNo tool selected")
        return "I couldn't find relevant information to answer your question.", []

In [ ]:
# Validation for CODE_BLOCK_11 and 12
def validate_code_block_11_12(agent_func):
    try:
        # Test the agent
        test_query = "What is the DeepSeek model architecture?"
        answer, images = agent_func(test_query)
        
        if not answer or answer == "I couldn't find relevant information to answer your question.":
            return ValidationResult(
                False,
                "Agent didn't return a proper answer.",
                hints=[
                    "CODE_BLOCK_11: Use select_tool([user_query])",
                    "CODE_BLOCK_12: Use get_information_for_question_answering(**tool_call.args)"
                ]
            )
        
        if len(images) == 0:
            return ValidationResult(
                False,
                "No images were retrieved.",
                hints=["Check that tool execution returns image paths"]
            )
        
        return ValidationResult(
            True,
            f"Fantastic! Your agent successfully answered the question and retrieved {len(images)} images."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error testing agent: {str(e)}",
            hints=["Check both CODE_BLOCK_11 and CODE_BLOCK_12"]
        )

display(create_validation_widget(11, validate_code_block_11_12, 'agent'))

## Step 12: Add Conversation Memory

In [ ]:
# Create collection for chat history
history_collection = db["history"]
history_collection.delete_many({})  # Clear existing history

# CODE_BLOCK_13: Create an index on session_id for efficient queries
# Hint: Use history_collection.create_index() with "session_id"
<CODE_BLOCK_13>

print("History collection configured")

In [ ]:
# Validation for CODE_BLOCK_13
def validate_code_block_13(collection):
    try:
        # Check indexes
        indexes = list(collection.list_indexes())
        index_names = [idx['key'] for idx in indexes]
        
        # Look for session_id index
        has_session_index = any('session_id' in str(keys) for keys in index_names)
        
        if not has_session_index:
            return ValidationResult(
                False,
                "No index found on session_id field.",
                hints=["Use history_collection.create_index('session_id')"]
            )
        
        return ValidationResult(
            True,
            "Great! Index created on session_id for efficient history queries."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error checking index: {str(e)}",
            hints=["Make sure the index was created"]
        )

display(create_validation_widget(13, validate_code_block_13, 'history_collection'))

In [ ]:
# Helper functions for memory
def store_chat_message(session_id: str, role: str, message_type: str, content: str):
    """Store a chat message in MongoDB"""
    message = {
        "session_id": session_id,
        "role": role,
        "type": message_type,
        "content": content,
        "timestamp": datetime.now()
    }
    
    # CODE_BLOCK_14: Insert the message into the history collection
    # Hint: Use history_collection.insert_one()
    <CODE_BLOCK_14>
    
def retrieve_session_history(session_id: str):
    """Retrieve chat history for a session"""
    # CODE_BLOCK_15: Query and sort messages by timestamp
    # Hint: Use find() with session_id filter and sort() by timestamp ascending (1)
    cursor = <CODE_BLOCK_15>
    
    messages = []
    for msg in cursor:
        if msg['type'] == 'text':
            messages.append(msg['content'])
        elif msg['type'] == 'image':
            from PIL import Image
            messages.append(Image.open(msg['content']))
    
    return messages

# Test the functions
test_session = "test_session_123"
store_chat_message(test_session, "user", "text", "Hello!")
store_chat_message(test_session, "agent", "text", "Hi there!")

history = retrieve_session_history(test_session)
print(f"Retrieved {len(history)} messages from history")

In [ ]:
# Validation for CODE_BLOCK_14 and 15
def validate_code_block_14_15(store_func, retrieve_func):
    try:
        # Test storing and retrieving
        test_session = f"validation_test_{datetime.now().timestamp()}"
        
        # Store messages
        store_func(test_session, "user", "text", "Test message 1")
        store_func(test_session, "agent", "text", "Test response 1")
        
        # Check if messages were stored
        count = history_collection.count_documents({"session_id": test_session})
        if count != 2:
            return ValidationResult(
                False,
                f"Expected 2 messages stored, but found {count}.",
                hints=["CODE_BLOCK_14: Use history_collection.insert_one(message)"]
            )
        
        # Retrieve messages
        retrieved = retrieve_func(test_session)
        
        if len(retrieved) != 2:
            return ValidationResult(
                False,
                f"Expected 2 messages retrieved, but got {len(retrieved)}.",
                hints=[
                    "CODE_BLOCK_15: Use history_collection.find({'session_id': session_id})",
                    "Don't forget to sort by timestamp: .sort('timestamp', 1)"
                ]
            )
        
        if retrieved != ["Test message 1", "Test response 1"]:
            return ValidationResult(
                False,
                "Messages not in correct order.",
                hints=["Make sure to sort by timestamp ascending (1)"]
            )
        
        return ValidationResult(
            True,
            "Excellent! Memory functions are working correctly."
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error testing memory functions: {str(e)}",
            hints=["Check both CODE_BLOCK_14 and CODE_BLOCK_15"]
        )

# Combined validation
memory_test = lambda: validate_code_block_14_15(store_chat_message, retrieve_session_history)
display(create_validation_widget(14, memory_test, 'store_chat_message'))

## Step 13: Create Memory-Enhanced Agent

In [ ]:
def agent_with_memory(user_query: str, session_id: str):
    """Agent with conversation memory"""
    print(f"\n{'='*50}")
    print(f"Session: {session_id}")
    print(f"User: {user_query}")
    
    # CODE_BLOCK_16: Retrieve conversation history
    # Hint: Use retrieve_session_history() with session_id
    history = <CODE_BLOCK_16>
    
    # CODE_BLOCK_17: Store the user's query
    # Hint: Use store_chat_message() with role="user", type="text"
    <CODE_BLOCK_17>
    
    # Prepare messages with history
    messages = history + [user_query]
    
    # Select and execute tool
    tool_call = select_tool(messages)
    
    if tool_call:
        print(f"\nSelected tool: {tool_call.name}")
        tool_images = get_information_for_question_answering(**tool_call.args)
        print(f"Retrieved {len(tool_images)} images")
        
        # CODE_BLOCK_18: Store retrieved images in history
        # Hint: Loop through tool_images and store each with role="user", type="image"
        for image in tool_images:
            <CODE_BLOCK_18>
        
        # Load images
        from PIL import Image
        images = [Image.open(img_path) for img_path in tool_images]
        
        # Generate answer with full context
        full_messages = history + [user_query] + images
        response = gemini_client.models.generate_content(
            model=LLM,
            contents=full_messages,
        )
        
        answer = response.text
        
        # CODE_BLOCK_19: Store agent's response
        # Hint: Use store_chat_message() with role="agent", type="text"
        <CODE_BLOCK_19>
        
        print(f"\nAgent: {answer}")
        
        # Display images
        for img_path in tool_images:
            display(IPImage(filename=img_path, width=400))
        
        return answer, tool_images
    else:
        answer = "I couldn't find relevant information to answer your question."
        store_chat_message(session_id, "agent", "text", answer)
        print(f"\nAgent: {answer}")
        return answer, []

In [ ]:
# Validation for CODE_BLOCK_16-19
def validate_code_block_16_19(agent_func):
    try:
        # Test with a new session
        test_session = f"validation_{datetime.now().timestamp()}"
        
        # First query
        answer1, images1 = agent_func("What are the key findings?", test_session)
        
        # Check if messages were stored
        messages = list(history_collection.find({"session_id": test_session}).sort("timestamp", 1))
        
        # Should have: user query, images (if any), agent response
        if len(messages) < 2:  # At least user and agent messages
            return ValidationResult(
                False,
                f"Expected at least 2 messages in history, found {len(messages)}.",
                hints=[
                    "CODE_BLOCK_16: retrieve_session_history(session_id)",
                    "CODE_BLOCK_17: store_chat_message(session_id, 'user', 'text', user_query)",
                    "CODE_BLOCK_18: store_chat_message(session_id, 'user', 'image', image)",
                    "CODE_BLOCK_19: store_chat_message(session_id, 'agent', 'text', answer)"
                ]
            )
        
        # Second query to test memory
        answer2, images2 = agent_func("Can you elaborate on that?", test_session)
        
        # Check if history is being used
        final_messages = list(history_collection.find({"session_id": test_session}))
        
        if len(final_messages) < 4:  # Should have messages from both queries
            return ValidationResult(
                False,
                "History doesn't seem to be accumulating correctly.",
                hints=["Make sure all 4 CODE_BLOCKs (16-19) are implemented"]
            )
        
        return ValidationResult(
            True,
            "🎉 Congratulations! Your memory-enhanced agent is working perfectly!"
        )
    except Exception as e:
        return ValidationResult(
            False,
            f"Error testing agent: {str(e)}",
            hints=["Check all CODE_BLOCKs 16-19"]
        )

display(create_validation_widget(16, validate_code_block_16_19, 'agent_with_memory'))

## 🎊 Final Test: Your Complete Agent

In [ ]:
# Test your complete agent!
session_id = f"demo_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Have a conversation
print("🤖 Let's test your multimodal agent with memory!\n")

# First question
agent_with_memory("What are the main contributions of the DeepSeek paper?", session_id)

# Follow-up question (tests memory)
agent_with_memory("Can you tell me more about their scaling laws?", session_id)

# Another follow-up
agent_with_memory("How does this compare to other models?", session_id)

## 🏆 Lab Completion

In [ ]:
# Check final progress
lab_progress.display_progress()

if len(lab_progress.completed_blocks) == 19:
    display(HTML("""
    <div style="
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 40px;
        border-radius: 20px;
        text-align: center;
        margin: 30px 0;
        box-shadow: 0 10px 25px rgba(0,0,0,0.2);
    ">
        <h1 style="margin: 0; font-size: 72px;">🏆</h1>
        <h1 style="margin: 20px 0; font-size: 36px;">Lab Champion!</h1>
        <p style="margin: 20px 0; font-size: 20px;">You've successfully completed the Multimodal Agents Lab!</p>
        <div style="background: rgba(255,255,255,0.2); padding: 20px; border-radius: 10px; margin: 20px 0;">
            <h3 style="margin: 10px 0;">🎯 You've learned how to:</h3>
            <ul style="text-align: left; display: inline-block; margin: 10px 0;">
                <li>Process PDFs and extract multimodal data</li>
                <li>Create vector embeddings and perform semantic search</li>
                <li>Build AI agents with tool calling capabilities</li>
                <li>Implement conversation memory with MongoDB</li>
                <li>Create production-ready multimodal AI systems</li>
            </ul>
        </div>
        <p style="margin: 20px 0; font-size: 18px; opacity: 0.9;">
            Time to build something amazing! 🚀
        </p>
    </div>
    """))
else:
    remaining = 19 - len(lab_progress.completed_blocks)
    display(HTML(f"""
    <div style="
        background-color: #fff3cd;
        border: 1px solid #ffeaa7;
        color: #856404;
        padding: 20px;
        border-radius: 10px;
        margin: 20px 0;
    ">
        <h3>Almost there! 💪</h3>
        <p>You have {remaining} CODE_BLOCK(s) remaining.</p>
        <p>Keep going - you're doing great!</p>
    </div>
    """))

## 🎯 Next Steps

Congratulations on completing the lab! Here are some ideas to extend your learning:

1. **Try Different PDFs**: Process other research papers or documents
2. **Enhance the Agent**: Add more tools like web search or calculations
3. **Implement ReAct**: Extend to the full ReAct (Reasoning + Acting) pattern
4. **Build an App**: Create a Streamlit or Gradio interface for your agent
5. **Add More Modalities**: Include audio or video processing

Share your creations and get help in the MongoDB community forums!